# 02 — Data Cleaning
**Objectif** : Nettoyer les données, convertir les dates, créer les variables métier et exporter les CSV propres.

Les fichiers nettoyés seront sauvegardés dans `data/cleaned/`.

In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH     = '../data/raw/'
CLEANED_PATH = '../data/cleaned/'
os.makedirs(CLEANED_PATH, exist_ok=True) # Crée le dossier "cleaned" s'il n'existe pas déjà

print('Dossiers prêts.')

Dossiers prêts.


## 1. Chargement des données brutes

In [2]:
orders       = pd.read_csv(RAW_PATH + 'olist_orders_dataset.csv')
items        = pd.read_csv(RAW_PATH + 'olist_order_items_dataset.csv')
customers    = pd.read_csv(RAW_PATH + 'olist_customers_dataset.csv')
reviews      = pd.read_csv(RAW_PATH + 'olist_order_reviews_dataset.csv')
payments     = pd.read_csv(RAW_PATH + 'olist_order_payments_dataset.csv')
products     = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')
sellers      = pd.read_csv(RAW_PATH + 'olist_sellers_dataset.csv')
translations = pd.read_csv(RAW_PATH + 'product_category_name_translation.csv')

print('Données brutes chargées.')

Données brutes chargées.


## 2. Nettoyage — Orders

In [7]:
print('Doublons order_id :', orders['order_id'].duplicated().sum())
print('Valeurs manquantes avant nettoyage :')
print(orders.isnull().sum())

# Conversion des colonnes dates
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Création des variables métier
orders['actual_delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

orders['is_late'] = (orders['delivery_delay_days'] > 0).astype(int)

orders['purchase_year']  = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.month
orders['purchase_quarter'] = orders['order_purchase_timestamp'].dt.quarter
orders['purchase_dayofweek'] = orders['order_purchase_timestamp'].dt.dayofweek

orders['carrier_delay_days'] = (
    orders['order_delivered_carrier_date'] - orders['order_purchase_timestamp']
).dt.days

print('\nVariables créées :')
display(orders[['order_id', 'actual_delivery_days', 'delivery_delay_days', 'is_late']].head())
print('\nDimensions finales :', orders.shape)

Doublons order_id : 0
Valeurs manquantes avant nettoyage :
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
actual_delivery_days             2965
delivery_delay_days              2965
is_late                             0
purchase_year                       0
purchase_month                      0
purchase_quarter                    0
purchase_dayofweek                  0
carrier_delay_days               1783
dtype: int64

Variables créées :


,order_id,actual_delivery_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0,0



Dimensions finales : (99441, 16)


## 3. Nettoyage — Reviews

In [ ]:
print('Doublons review_id :', reviews['review_id'].duplicated().sum())
print('Valeurs manquantes :')
print(reviews.isnull().sum())

# Conversion des dates
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# Supprimer les doublons (garder le dernier avis par commande)
reviews_clean = reviews.sort_values('review_creation_date').drop_duplicates(
    subset='order_id', keep='last'
).copy()

# Remplir les commentaires manquants
reviews_clean['review_comment_title']   = reviews_clean['review_comment_title'].fillna('')
reviews_clean['review_comment_message'] = reviews_clean['review_comment_message'].fillna('')

# Délai de réponse à l'avis (en heures)
reviews_clean['response_delay_hours'] = (
    reviews_clean['review_answer_timestamp'] - reviews_clean['review_creation_date']
).dt.total_seconds() / 3600

print(f'\nAvis avant dédup : {len(reviews):,}') #avis avant déduplication: dataframe reviews
print(f'Avis après dédup : {len(reviews_clean):,}') #avis après déduplication: dataframe reviews_clean
print('Dimensions finales :', reviews_clean.shape)

Doublons review_id : 814
Valeurs manquantes :
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Avis avant dédup : 99,224
Avis après dédup : 98,673
Dimensions finales : (98673, 8)


## 4. Nettoyage — Products

In [10]:
print('Doublons product_id :', products['product_id'].duplicated().sum())
print('Valeurs manquantes :')
print(products.isnull().sum())

# Traduction des catégories
products_clean = products.merge(translations, on='product_category_name', how='left')

# Remplir les catégories manquantes
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english'].fillna('unknown')
)
products_clean['product_category_name'] = (
    products_clean['product_category_name'].fillna('unknown')
)

# Imputation des valeurs numériques manquantes par la médiane
num_cols = ['product_name_lenght', 'product_description_lenght',
            'product_photos_qty', 'product_weight_g',
            'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in num_cols:
    median_val = products_clean[col].median()
    products_clean[col] = products_clean[col].fillna(median_val)
    print(f'  {col}: {products[col].isnull().sum()} manquants → imputés par médiane ({median_val:.1f})')

# Volume approximatif du produit (cm³)
products_clean['product_volume_cm3'] = (
    products_clean['product_length_cm'] *
    products_clean['product_height_cm'] *
    products_clean['product_width_cm']
)

print('\nDimensions finales :', products_clean.shape)

Doublons product_id : 0
Valeurs manquantes :
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
  product_name_lenght: 610 manquants → imputés par médiane (51.0)
  product_description_lenght: 610 manquants → imputés par médiane (595.0)
  product_photos_qty: 610 manquants → imputés par médiane (1.0)
  product_weight_g: 2 manquants → imputés par médiane (700.0)
  product_length_cm: 2 manquants → imputés par médiane (25.0)
  product_height_cm: 2 manquants → imputés par médiane (13.0)
  product_width_cm: 2 manquants → imputés par médiane (20.0)

Dimensions finales : (32951, 11)


## 5. Nettoyage — Items

In [17]:
print('Valeurs manquantes items :', items.isnull().sum().sum())
print('Doublons :', items.duplicated().sum())

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

# Prix total par ligne (prix + frais de port)
items['total_value'] = items['price'] + items['freight_value']

# Ratio frais de port / prix
items['freight_ratio'] = (items['freight_value'] / items['price']).round(4)

print('Dimensions finales :', items.shape)

Valeurs manquantes items : 0
Doublons : 0
Dimensions finales : (112650, 9)


## 6. Nettoyage — Customers & Sellers

In [18]:
# Nettoyage minimal — peu de problèmes dans ces tables
print('Doublons customers :', customers.duplicated().sum())
print('Doublons sellers :', sellers.duplicated().sum())
print('Manquants customers :', customers.isnull().sum().sum())
print('Manquants sellers :', sellers.isnull().sum().sum())

# Normalisation des noms de ville en minuscule
customers['customer_city'] = customers['customer_city'].str.lower().str.strip()
sellers['seller_city']     = sellers['seller_city'].str.lower().str.strip()

print('\nTop 10 villes clients :')
print(customers['customer_city'].value_counts().head(10))

Doublons customers : 0
Doublons sellers : 0
Manquants customers : 0
Manquants sellers : 0

Top 10 villes clients :
customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64


## 7. Nettoyage — Payments

In [22]:
print('Valeurs manquantes payments :', payments.isnull().sum().sum())
print('Types de paiement :', payments['payment_type'].unique())

# Supprimer les lignes 'not_defined'
payments_clean = payments[payments['payment_type'] != 'not_defined'].copy()

# Valeur totale par commande (somme des paiements séquentiels)
payment_totals = payments_clean.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    nb_installments=('payment_installments', 'max')
).reset_index()

print(f'\nLignes supprimées (not_defined) : {len(payments) - len(payments_clean)}')
print('Dimensions payments_clean :', payments_clean.shape)
print('Dimensions payment_totals :', payment_totals.shape)

Valeurs manquantes payments : 0
Types de paiement : ['credit_card' 'boleto' 'voucher' 'debit_card' 'not_defined']

Lignes supprimées (not_defined) : 3
Dimensions payments_clean : (103883, 5)
Dimensions payment_totals : (99437, 3)


## 8. Export des fichiers nettoyés

In [23]:
exports = {
    'orders_clean.csv':        orders,
    'items_clean.csv':         items,
    'customers_clean.csv':     customers,
    'reviews_clean.csv':       reviews_clean,
    'payment_totals.csv':      payment_totals,
    'products_clean.csv':      products_clean,
    'sellers_clean.csv':       sellers,
}

for filename, df in exports.items():
    path = CLEANED_PATH + filename
    df.to_csv(path, index=False)
    print(f'  Exporté : {filename} → {len(df)} lignes')

print('\nTous les fichiers nettoyés ont été exportés dans data/cleaned/')

  Exporté : orders_clean.csv → 99441 lignes
  Exporté : items_clean.csv → 112650 lignes
  Exporté : customers_clean.csv → 99441 lignes
  Exporté : reviews_clean.csv → 98673 lignes
  Exporté : payment_totals.csv → 99437 lignes
  Exporté : products_clean.csv → 32951 lignes
  Exporté : sellers_clean.csv → 3095 lignes

Tous les fichiers nettoyés ont été exportés dans data/cleaned/
